# Trabajo final · Data Analyst · Rutas del Plata

**Transporte interurbano de pasajeros** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Cómo recuperamos la ocupación que cayó en el último trimestre?**

**Decisión que habilita:** Dónde reforzar la acción comercial y qué canal priorizar.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Rutas del Plata

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNp1Vttu4zYQfc9XEHqSsgpX8nXj1AWaFvtWoNgu+iIYBiPSMTeSqFKS0yTIx/QD+lDsJ+THeoaUHNtNBFuiSM7wzOGZEXVZ"
    "G9uyqivrByYaVtUxq0Ul0cSvlmdSbdi66XJtQhuzRlmtYrYTVouqVQ0mR4szhqthSz/Kc1M/hBEXTftQq9DcfFN5G12xEhMs"
    "t/BtyrBQVdhEEfuB1d46K1cYz/aOw13MbMQ2xrId05WbsLpiVrWdxduZx1WKh65Zq6YWhM+ZLIY52Y4X5l7ZMAJe3tV132Qf"
    "WMCCmG5o7laZ5Ror3irbhEnMJtHKO5ciXduuFc1aqmJdF6IVYaOUXKZJ2sdsAbmq+5g4jERXtGtb3bqJiDnXnRRSETdZ8MU0"
    "iM7Q0j+/fLfS3Ahq/6pg/eibwjIsxn6jxajja5d35cvfFbV/F4XvvBbbl38Euy5ElQs/UrWCfVYBCLojmrNxgkjmdEuTKQXl"
    "miNqplNqz1zvPFn5OChM2NWS/4KVP1tRqvAp0NIRECxYtgm+POlFMpLPgdsTTXuCuG9VmMbsMloBhrH6VlWYHVx3qjIN+0lb"
    "1RBAMNDqymBoIASddyXe78pn0OTWz4IW9GzE+kY0CAVwQtfPEdI5S3iaYrvSTxG3pqtkmEQOOk3GVKnzNnzUdW/SI49ZxueT"
    "mPE5ouWzGbU+UWuE25Rac2rNJqsIKKS5h6enZMH4Jaam9MTwqH+OFyzlxBo9U3RM+46Z60ier9hOi28UG8JWducozcBMhn3R"
    "coc3LUkyiUNONEqiEaxL0aq1pzMYJaPpRZLiR9S5t3R0MU6DXnR0/UnsSF6aqt2yC5ZG7ONHNiZ+9lPIve3cNh1S8uqDLpPn"
    "8EQcZrZbgWZwkBGcB7O5V+pudTRbb8jjcsmCL8kkYFA9AVkSI+ToHJGB5HdNxm+apHyUvGsye8ckHb1rkr5jkhwD2yHraEOq"
    "NrT8BuIstSjCCfZ9UxjRhkjrvICeyP4D6lZlbIkJpKNkgjLC0ylul/MoctJRpbEC/vbGXqQWJei2N035ZQyCLmHgAA+lkIog"
    "uudTpgpo2Xs4qEmpS9VRQuuQij4sD3aZrhx1AEVjeeoSyhkkyQXKH/CEm+CPJzhZJFNkMoTaxUziwfOt0bkKs6AUL/+KypUV"
    "pKNU1KhMvkVGotYvMz6muP0tWYEIoozIjHsSMN8DEtIEFGgPzwUXWCUK/UhDUXQURKE2LUIgT1cMK++59OotTJ4d6Zi2l9Af"
    "Fw2uaSJwHfm+3+pC+RV+ZMlxBtAleiGUugpp1iEhqG6U+33wVEHGtPkrt+3O5cWSiSuX2bQzVxSv24s9peCeWLwxhWqVRemm"
    "N4FSmWsxsDrZszo6RU9XTS6fnKcFC/knh4FKVzKNjl3T8KUbmQxS3a/lxsg0Gfux5wxgUZxQno8hk2pevvu9Xzdbcx8cbGsP"
    "uq7ewNm+7tsphZAKamYfr0dHFTil29jdRiufGqHDg2zuYbiEPhI3CnrkFYVUq0T1PyBDBT4Q/jWEbxfJzAn/JA8QWswgXZ+1"
    "AoWQJHjOQiRmr2EHaGDSr42ye76PdshyR28E2hP8EQjibXut+1w8/c4OH43cFF1ZNcuMvruuk0gfvsFoblS+9YmJ4407RohG"
    "q6o1zbo1rSj8t3bfR6mkpek/wJSaawic3oCKMnBFX97XL9URqKH/BFbf3QPbgwQ9ojhc3E1wJ0t1sCBadYejCox6jcuuJuEN"
    "m9WIsi5USLlIB8Shn2jmSUpS9xpYw1+rlv0h6ziG3OBU1IbZawRYBKrTVIfVWldS/bX8ajsVDcttrMiX6Vu+OTlpvU0oram9"
    "4euSWR86nVZeD8n7eNxgfHJM9QLx5y5/Un0KXGlDfrpn7IWvqGMQRzD4pEl98/nsDHJU1hXLN06rZ/8B8OlvZA=="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La ocupación por trimestre y por ruta (JOIN de `viajes` con `rutas`).
2. La tasa de no-show y el ingreso por canal (JOIN de `reservas` con `viajes`).
3. Un control de integridad: ¿los asientos vendidos de cada viaje coinciden con la suma de sus reservas?

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`demora_min`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (mes o trimestre, turno, día de la semana, ocupación del viaje, ingreso por reserva).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: ocupación (asientos vendidos sobre asientos totales de los viajes realizados), ingreso por asiento, tasa de no-show y puntualidad.

**Segmentación:** ruta, canal, turno y trimestre.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.